# TDGraph-IDS -- Phase 2: Preprocessing and Behavioral DPI Feature Engineering

**Goal:** Transform the unified dataset into ML-ready feature matrices.
This phase engineers 16 behavioral DPI features from flow metadata,
applies SMOTE to balance the training set, and produces clean train/test
splits with no data leakage.

**Input:** `data/processed/unified_flows.csv`

**Outputs:**
- `data/processed/X_train.csv` -- 40-feature scaled training matrix (SMOTE-balanced)
- `data/processed/X_test.csv`  -- 40-feature scaled test matrix (real distribution)
- `data/processed/y_train.csv` -- binary training labels
- `data/processed/y_test.csv`  -- binary test labels
- `data/processed/y_train_multi.csv` -- multiclass training labels
- `data/processed/y_test_multi.csv`  -- multiclass test labels
- `data/processed/scaler.pkl`        -- fitted StandardScaler (train-only)
- `data/processed/label_encoder.pkl` -- fitted LabelEncoder
- `data/processed/feature_columns.csv` -- ordered feature names
- `data/graph/graph_data.csv`        -- IP-level flows for Phase 3
- `outputs/plots/phase2_*.png`       -- submission visualizations

**Key design decisions:**
- 16 behavioral DPI features engineered from flow metadata (no raw packets needed).
- SMOTE applied on training set only after splitting -- no leakage.
- StandardScaler fitted on training data only -- no leakage.
- Stratified 80/20 split preserves 3.51% attack rate in both splits.
- IP columns preserved in graph_data.csv before being dropped from ML features.

---

## 2.1 -- Setup

In [14]:
import sys
import os
import warnings
import logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import CFG, ensure_dirs

ensure_dirs()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('phase2')

log.info('Project root : %s', CFG.ROOT)
log.info('Input        : %s', CFG.UNIFIED)
log.info('Graph dir    : %s', CFG.DATA_GRAPH)

08:12:54  INFO  Project root : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS
08:12:54  INFO  Input        : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\data\processed\unified_flows.csv
08:12:54  INFO  Graph dir    : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\data\graph


## 2.2 -- Load unified dataset

Load Phase 1 output and validate required columns are present.

In [15]:
def load_unified_dataset(path: str) -> pd.DataFrame:
    """
    Load unified_flows.csv produced by Phase 1.

    Parameters
    ----------
    path : str
        Path to unified_flows.csv.

    Returns
    -------
    pd.DataFrame
        Loaded dataset.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    ValueError
        If required columns are absent.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'unified_flows.csv not found at {path}. Run Phase 1 first.'
        )

    log.info('Loading unified_flows.csv ...')
    df = pd.read_csv(path, low_memory=False)
    log.info('  Shape: %d rows x %d columns', *df.shape)

    required = {'src_ip', 'dst_ip', 'label', 'attack_unified', 'source'}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(
            f'Required columns missing: {missing}. Re-run Phase 1.'
        )

    return df


df = load_unified_dataset(str(CFG.UNIFIED))

print('Shape   :', df.shape)
print('Columns :', df.columns.tolist())
print()
print('Label distribution:')
print(df['label'].value_counts().to_string())
print('Attack rate: {:.2f}%'.format(df['label'].mean() * 100))

08:13:11  INFO  Loading unified_flows.csv ...
08:13:25  INFO    Shape: 2365636 rows x 35 columns


Shape   : (2365636, 35)
Columns : ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'l7_proto', 'fwd_bytes', 'fwd_pkts', 'bwd_bytes', 'bwd_pkts', 'duration_ms', 'tcp_flags', 'client_tcp_flags', 'server_tcp_flags', 'duration_in', 'duration_out', 'min_ttl', 'max_ttl', 'label', 'attack_raw', 'attack_unified', 'source', 'flow_bytes_per_sec', 'flow_pkts_per_sec', 'iat_mean', 'iat_std', 'iat_max', 'iat_min', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'init_fwd_win', 'init_bwd_win']

Label distribution:
label
0    2282659
1      82977
Attack rate: 3.51%


## 2.3 -- Preserve graph-compatible flow data

Save IP-level columns for Phase 3 graph construction before any preprocessing
modifies or removes them. This step runs first so no downstream operation
can corrupt the graph input.

In [16]:
def save_graph_data(df: pd.DataFrame) -> None:
    """
    Extract and save flow columns needed for Phase 3 graph construction.

    Saves IP addresses, port numbers, traffic volumes, protocol, and labels.
    All preprocessing operates on a separate copy of the data.

    Parameters
    ----------
    df : pd.DataFrame
        Full unified dataset from Phase 1.
    """
    graph_cols = [
        'src_ip', 'dst_ip', 'src_port', 'dst_port',
        'fwd_bytes', 'fwd_pkts', 'bwd_bytes', 'bwd_pkts',
        'duration_ms', 'protocol', 'label', 'attack_unified', 'source',
    ]
    available = [c for c in graph_cols if c in df.columns]
    missing   = [c for c in graph_cols if c not in df.columns]

    if missing:
        log.warning('Graph columns not found in dataset: %s', missing)

    graph_df = df[available].copy()
    out_path = CFG.DATA_GRAPH / 'graph_data.csv'
    graph_df.to_csv(out_path, index=False)

    log.info(
        'Saved graph_data.csv: %d rows x %d columns  (%.1f MB)',
        *graph_df.shape,
        out_path.stat().st_size / 1e6,
    )


save_graph_data(df)
print('graph_data.csv saved to data/graph/')

08:13:49  INFO  Saved graph_data.csv: 2365636 rows x 13 columns  (186.9 MB)


graph_data.csv saved to data/graph/


## 2.4 -- Behavioral DPI Feature Engineering

16 behavioral features are derived from flow metadata.
No raw packet inspection is required -- all signals come from
statistics available in both datasets.

| Group | Features | What they detect |
|---|---|---|
| Payload | byte_entropy, pkt_size_mean, pkt_size_ratio | Encrypted payloads, scanning, tunneling |
| Timing | iat_variance, iat_cv, burst_score | Bot regularity, DoS flooding |
| Direction | flow_efficiency, direction_asymmetry | SYN floods, scan patterns |
| Protocol | protocol_anomaly, syn_ratio, rst_ratio | C2 tunneling, port scanning |
| Port | is_ephemeral_src, is_well_known_dst, is_suspicious_port, flow_bytes_per_sec, flow_pkts_per_sec | Backdoors, service targeting |

In [17]:
def engineer_dpi_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Engineer 16 behavioral DPI proxy features from flow-level statistics.

    All features are derived from columns available in both datasets
    without requiring raw packet payload inspection.

    Parameters
    ----------
    df : pd.DataFrame
        Unified dataset with fwd_bytes, bwd_bytes, fwd_pkts, bwd_pkts,
        duration_ms, src_port, dst_port, protocol columns.

    Returns
    -------
    pd.DataFrame
        Original dataframe with 16 new DPI feature columns appended.
    """
    df = df.copy()

    # Ensure numeric types and safe denominators
    for col in ['fwd_bytes', 'bwd_bytes', 'fwd_pkts', 'bwd_pkts',
                'duration_ms', 'src_port', 'dst_port', 'protocol']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    total_bytes = df['fwd_bytes'] + df['bwd_bytes'] + 1e-9
    total_pkts  = df['fwd_pkts']  + df['bwd_pkts']  + 1e-9
    duration_ms = df['duration_ms'].clip(lower=0.001)
    duration_s  = duration_ms / 1000.0

    # ── Group 1: Payload analysis ──────────────────────────────────

    # byte_entropy: Shannon entropy proxy from fwd/bwd byte ratio.
    # Encrypted/obfuscated traffic has near-equal fwd/bwd bytes
    # (high entropy). Scanning has extreme imbalance (low entropy).
    fwd_ratio = (df['fwd_bytes'] + 1) / total_bytes
    bwd_ratio = (df['bwd_bytes'] + 1) / total_bytes
    df['byte_entropy'] = -(fwd_ratio * np.log2(fwd_ratio + 1e-9) +
                           bwd_ratio * np.log2(bwd_ratio + 1e-9))

    # pkt_size_mean: Average bytes per packet.
    # Scanning: tiny packets (40-60 bytes). Exfiltration: large (near MTU).
    df['pkt_size_mean'] = total_bytes / total_pkts

    # pkt_size_ratio: Forward to backward packet size asymmetry.
    # High ratio indicates mixed traffic typical of protocol tunneling.
    df['pkt_size_ratio'] = (df['fwd_bytes'] + 1) / (df['bwd_bytes'] + 1)

    # ── Group 2: Timing and burst behaviour ────────────────────────

    # iat_variance: Inter-arrival time variance proxy.
    # Bot and automated tools send at perfectly regular intervals
    # (near-zero variance). Human traffic shows high variance.
    if 'iat_std' in df.columns and 'iat_mean' in df.columns:
        iat_std  = pd.to_numeric(df['iat_std'],  errors='coerce').fillna(0)
        iat_mean = pd.to_numeric(df['iat_mean'], errors='coerce').fillna(1)
        df['iat_variance'] = iat_std ** 2
        df['iat_cv']       = iat_std / (iat_mean + 1e-9)
    else:
        # Estimate from flow duration and packet count
        df['iat_variance'] = duration_ms / total_pkts
        df['iat_cv']       = df['iat_variance'] / (duration_ms + 1e-9)

    # burst_score: Bytes per millisecond.
    # DoS flooding sends gigabytes in milliseconds -- extremely high score.
    df['burst_score'] = total_bytes / duration_ms

    # ── Group 3: Flow directionality ───────────────────────────────

    # flow_efficiency: Fraction of traffic that is forward (src to dst).
    # Value near 1.0 = almost entirely one-directional = SYN flood.
    # Normal communication is near 0.5 (bidirectional).
    df['flow_efficiency'] = df['fwd_bytes'] / total_bytes

    # direction_asymmetry: Packet count imbalance between directions.
    # Port scans send many packets but get few responses.
    df['direction_asymmetry'] = (
        (df['fwd_pkts'] - df['bwd_pkts']).abs() / total_pkts
    )

    # ── Group 4: Protocol behaviour ────────────────────────────────

    # protocol_anomaly: Flag flows where protocol does not match expected port.
    # Attackers hide C2 traffic inside legitimate-looking protocols on wrong ports.
    EXPECTED_PORT_PROTOCOL = {
        80: 6, 443: 6, 22: 6, 21: 6,
        53: 17, 123: 17, 161: 17,
    }
    dst_port = df['dst_port'].astype(int)
    protocol = df['protocol'].astype(int)
    anomaly  = pd.Series(0, index=df.index)
    for port, expected_proto in EXPECTED_PORT_PROTOCOL.items():
        mask = (dst_port == port) & (protocol != expected_proto)
        anomaly = anomaly | mask.astype(int)
    df['protocol_anomaly'] = anomaly

    # syn_ratio and rst_ratio: from TCP flag columns if available.
    # SYN flood: ratio near 1.0. Port scanning: high RST ratio.
    if 'syn_flag_cnt' in df.columns and 'ack_flag_cnt' in df.columns:
        total_flags = (
            pd.to_numeric(df.get('syn_flag_cnt', 0), errors='coerce').fillna(0) +
            pd.to_numeric(df.get('ack_flag_cnt', 0), errors='coerce').fillna(0) +
            pd.to_numeric(df.get('rst_flag_cnt', 0), errors='coerce').fillna(0) +
            pd.to_numeric(df.get('psh_flag_cnt', 0), errors='coerce').fillna(0) +
            1e-9
        )
        df['syn_ratio'] = (
            pd.to_numeric(df.get('syn_flag_cnt', 0), errors='coerce').fillna(0)
            / total_flags
        )
        df['rst_ratio'] = (
            pd.to_numeric(df.get('rst_flag_cnt', 0), errors='coerce').fillna(0)
            / total_flags
        )
    elif 'tcp_flags' in df.columns:
        # NF-UNSW provides aggregated TCP flags as integer bitmask
        flags    = pd.to_numeric(df['tcp_flags'], errors='coerce').fillna(0).astype(int)
        syn_flag = ((flags & 0x02) > 0).astype(float)
        rst_flag = ((flags & 0x04) > 0).astype(float)
        df['syn_ratio'] = syn_flag
        df['rst_ratio'] = rst_flag
    else:
        df['syn_ratio'] = 0.0
        df['rst_ratio'] = 0.0

    # ── Group 5: Port classification ───────────────────────────────

    # is_ephemeral_src: Source port above 49152 (IANA ephemeral range).
    # Attackers scripting connections use high ephemeral source ports.
    df['is_ephemeral_src'] = (df['src_port'] > 49152).astype(int)

    # is_well_known_dst: Destination port below 1024.
    # Attacks predominantly target well-known service ports.
    df['is_well_known_dst'] = (df['dst_port'] < 1024).astype(int)

    # is_suspicious_port: Known backdoor and C2 destination ports.
    SUSPICIOUS_PORTS = {
        4444,   # Metasploit default
        1337,   # Elite / leet port
        31337,  # Back Orifice
        6666,   # IRC botnet
        6667,   # IRC botnet
        9001,   # Tor
        8080,   # Common proxy / C2
        1080,   # SOCKS proxy
    }
    df['is_suspicious_port'] = dst_port.isin(SUSPICIOUS_PORTS).astype(int)

    # flow_bytes_per_sec: Throughput metric.
    # Combined with burst_score provides a strong DoS detection signal.
    df['flow_bytes_per_sec'] = total_bytes / duration_s

    # flow_pkts_per_sec: Packet rate.
    # SYN flood maximises this. High value with small pkt_size_mean = flooding.
    df['flow_pkts_per_sec'] = total_pkts / duration_s

    log.info('DPI feature engineering complete')
    log.info('  16 behavioral DPI features added')

    return df


DPI_FEATURE_COLS = [
    'byte_entropy', 'pkt_size_mean', 'pkt_size_ratio',
    'iat_variance', 'iat_cv', 'burst_score',
    'flow_efficiency', 'direction_asymmetry',
    'protocol_anomaly', 'syn_ratio', 'rst_ratio',
    'is_ephemeral_src', 'is_well_known_dst', 'is_suspicious_port',
    'flow_bytes_per_sec', 'flow_pkts_per_sec',
]

df = engineer_dpi_features(df)

dpi_present = [c for c in DPI_FEATURE_COLS if c in df.columns]
print('DPI features engineered: {}'.format(len(dpi_present)))
print('Features:', dpi_present)
print()
print('Sample DPI feature values (first 3 rows):')
print(df[dpi_present[:6]].head(3).round(4).to_string())

08:13:57  INFO  DPI feature engineering complete
08:13:57  INFO    16 behavioral DPI features added


DPI features engineered: 16
Features: ['byte_entropy', 'pkt_size_mean', 'pkt_size_ratio', 'iat_variance', 'iat_cv', 'burst_score', 'flow_efficiency', 'direction_asymmetry', 'protocol_anomaly', 'syn_ratio', 'rst_ratio', 'is_ephemeral_src', 'is_well_known_dst', 'is_suspicious_port', 'flow_bytes_per_sec', 'flow_pkts_per_sec']

Sample DPI feature values (first 3 rows):
   byte_entropy  pkt_size_mean  pkt_size_ratio  iat_variance  iat_cv  burst_score
0        0.2707        50.5000          0.0515           0.0     0.0     202000.0
1        0.9396        60.8333          0.5574           0.0     0.0     730000.0
2        0.9646        61.5500          0.6418           0.0     0.0    1231000.0


## 2.5 -- Define feature and target columns

40 total features: 24 raw flow features + 16 engineered DPI features.
Explicitly declaring the exclusion set prevents silent leakage.

In [18]:
EXCLUDE_FROM_FEATURES = {
    # Target columns
    'label', 'attack_raw', 'attack_unified',
    # IP and port columns -- used for graph, not ML
    'src_ip', 'dst_ip', 'src_port', 'dst_port',
    # Source tag -- would cause dataset leakage
    'source',
    # Raw TCP flag text columns
    'tcp_flags', 'client_tcp_flags', 'server_tcp_flags',
    # Node IDs if present
    'src_node', 'dst_node',
}

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
feature_cols = [
    c for c in numeric_cols
    if c not in EXCLUDE_FROM_FEATURES
]

# Classify features for reporting
flow_features = [c for c in feature_cols if c not in DPI_FEATURE_COLS]
dpi_features  = [c for c in feature_cols if c in DPI_FEATURE_COLS]

log.info('Flow-level features : %d', len(flow_features))
log.info('DPI features        : %d', len(dpi_features))
log.info('Total features      : %d', len(feature_cols))

print('Flow-level features ({} total):'.format(len(flow_features)))
for i, col in enumerate(flow_features, 1):
    print('  {:2d}. {}'.format(i, col))
print()
print('DPI features ({} total):'.format(len(dpi_features)))
for i, col in enumerate(dpi_features, 1):
    print('  {:2d}. {}'.format(i, col))
print()
print('Total features: {}'.format(len(feature_cols)))

08:14:29  INFO  Flow-level features : 22
08:14:29  INFO  DPI features        : 16
08:14:29  INFO  Total features      : 38


Flow-level features (22 total):
   1. protocol
   2. l7_proto
   3. fwd_bytes
   4. fwd_pkts
   5. bwd_bytes
   6. bwd_pkts
   7. duration_ms
   8. duration_in
   9. duration_out
  10. min_ttl
  11. max_ttl
  12. iat_mean
  13. iat_std
  14. iat_max
  15. iat_min
  16. syn_flag_cnt
  17. rst_flag_cnt
  18. psh_flag_cnt
  19. ack_flag_cnt
  20. urg_flag_cnt
  21. init_fwd_win
  22. init_bwd_win

DPI features (16 total):
   1. flow_bytes_per_sec
   2. flow_pkts_per_sec
   3. byte_entropy
   4. pkt_size_mean
   5. pkt_size_ratio
   6. iat_variance
   7. iat_cv
   8. burst_score
   9. flow_efficiency
  10. direction_asymmetry
  11. protocol_anomaly
  12. syn_ratio
  13. rst_ratio
  14. is_ephemeral_src
  15. is_well_known_dst
  16. is_suspicious_port

Total features: 38


## 2.6 -- Build feature matrix and target vectors

In [19]:
def build_feature_matrix(
    df: pd.DataFrame,
    feature_cols: list,
) -> tuple:
    """
    Build feature matrix and target vectors from the unified dataset.

    Parameters
    ----------
    df           : pd.DataFrame  Full unified dataset with DPI features.
    feature_cols : list          Ordered list of feature column names.

    Returns
    -------
    tuple of (X, y_binary, y_multi, label_encoder)
        X            : pd.DataFrame  Feature matrix (40 columns).
        y_binary     : pd.Series     Binary labels (0=Benign, 1=Attack).
        y_multi      : pd.Series     Integer-encoded attack category.
        label_encoder: LabelEncoder  Fitted encoder for attack categories.
    """
    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)
    X = X.clip(-1e9, 1e9)

    y_binary = df['label'].astype(int).copy()

    le = LabelEncoder()
    y_multi = pd.Series(
        le.fit_transform(df['attack_unified'].fillna('Other')),
        name='attack_label',
    )

    log.info('Feature matrix : %d rows x %d columns', *X.shape)
    log.info('Binary labels  : %s', dict(y_binary.value_counts()))
    log.info('Categories     : %d unique', le.classes_.size)

    return X, y_binary, y_multi, le


X, y_binary, y_multi, label_encoder = build_feature_matrix(df, feature_cols)

print('X shape    :', X.shape)
print('y_binary   :', dict(y_binary.value_counts()))
print('Categories :', list(label_encoder.classes_))

08:14:57  INFO  Feature matrix : 2365636 rows x 38 columns
08:14:57  INFO  Binary labels  : {0: np.int64(2282659), 1: np.int64(82977)}
08:14:57  INFO  Categories     : 8 unique


X shape    : (2365636, 38)
y_binary   : {0: np.int64(2282659), 1: np.int64(82977)}
Categories : ['Analysis', 'Benign', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Malware', 'Reconnaissance']


## 2.7 -- Log1p transform on skewed features

Features with absolute skewness above 2.0 receive log1p transformation.
This is applied before the train/test split because it is a deterministic
mathematical operation -- no statistics are learned from the data.

In [20]:
def apply_log_transform(X: pd.DataFrame, threshold: float = 2.0) -> tuple:
    """
    Apply log1p transformation to columns with absolute skewness above threshold.

    log1p(x) = log(x + 1) handles zero values without producing -inf.
    Negative values are clipped to 0 before transformation.

    Parameters
    ----------
    X         : pd.DataFrame  Feature matrix.
    threshold : float         Skewness magnitude above which to transform.

    Returns
    -------
    tuple of (X_transformed, skewed_cols)
    """
    skewness    = X.skew()
    skewed_cols = skewness[skewness.abs() > threshold].index.tolist()

    X = X.copy()
    X[skewed_cols] = X[skewed_cols].clip(lower=0).apply(np.log1p)

    log.info(
        'Log1p applied to %d / %d features (threshold abs skew > %.1f)',
        len(skewed_cols), X.shape[1], threshold,
    )
    return X, skewed_cols


X, skewed_cols = apply_log_transform(X, threshold=2.0)

print('Features log-transformed: {}'.format(len(skewed_cols)))
for col in skewed_cols:
    print('  ', col)

08:15:15  INFO  Log1p applied to 17 / 38 features (threshold abs skew > 2.0)


Features log-transformed: 17
   protocol
   l7_proto
   fwd_bytes
   fwd_pkts
   bwd_bytes
   bwd_pkts
   duration_ms
   duration_in
   duration_out
   min_ttl
   max_ttl
   flow_pkts_per_sec
   pkt_size_ratio
   burst_score
   direction_asymmetry
   protocol_anomaly
   is_suspicious_port


## 2.8 -- Stratified train / test split

80% train, 20% test. Stratified on binary label to preserve the
3.51% attack rate in both splits.

SMOTE is applied only after this split. Applying SMOTE before splitting
would place synthetic training samples in the test set, inflating all metrics.

In [21]:
def stratified_split(
    X: pd.DataFrame,
    y_binary: pd.Series,
    y_multi: pd.Series,
) -> tuple:
    """
    Perform a stratified 80/20 train/test split.

    Parameters
    ----------
    X        : pd.DataFrame  Feature matrix.
    y_binary : pd.Series     Binary labels.
    y_multi  : pd.Series     Multiclass integer labels.

    Returns
    -------
    tuple of (X_train, X_test, y_train, y_test, y_train_multi, y_test_multi)
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary,
        test_size=CFG.TEST_SIZE,
        random_state=CFG.RANDOM_STATE,
        stratify=y_binary,
    )

    y_train_multi = y_multi.iloc[y_train.index].reset_index(drop=True)
    y_test_multi  = y_multi.iloc[y_test.index].reset_index(drop=True)

    log.info('Train: %d rows  attack rate: %.2f%%',
             len(X_train), y_train.mean() * 100)
    log.info('Test : %d rows  attack rate: %.2f%%',
             len(X_test),  y_test.mean()  * 100)

    return X_train, X_test, y_train, y_test, y_train_multi, y_test_multi


X_train, X_test, y_train, y_test, y_train_multi, y_test_multi = \
    stratified_split(X, y_binary, y_multi)

print('X_train shape :', X_train.shape)
print('X_test  shape :', X_test.shape)
print('Train attack rate : {:.2f}%'.format(y_train.mean() * 100))
print('Test  attack rate : {:.2f}%'.format(y_test.mean()  * 100))
print()
print('Train label counts before SMOTE:')
print(y_train.value_counts().to_string())

08:15:27  INFO  Train: 1892508 rows  attack rate: 3.51%
08:15:27  INFO  Test : 473128 rows  attack rate: 3.51%


X_train shape : (1892508, 38)
X_test  shape : (473128, 38)
Train attack rate : 3.51%
Test  attack rate : 3.51%

Train label counts before SMOTE:
label
0    1826126
1      66382


## 2.9 -- StandardScaler (fit on train only)

Fitted exclusively on X_train. Applied to both splits.
Fitting on test data would expose test statistics to the training pipeline.

In [22]:
def fit_and_scale(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
) -> tuple:
    """
    Fit StandardScaler on training data only and transform both splits.

    Parameters
    ----------
    X_train : pd.DataFrame  Training feature matrix.
    X_test  : pd.DataFrame  Test feature matrix.

    Returns
    -------
    tuple of (X_train_scaled, X_test_scaled, scaler)
    """
    scaler = StandardScaler()

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )

    log.info('Scaler fitted on training data only')
    return X_train_scaled, X_test_scaled, scaler


X_train_scaled, X_test_scaled, scaler = fit_and_scale(X_train, X_test)

print('X_train_scaled shape :', X_train_scaled.shape)
print('X_test_scaled  shape :', X_test_scaled.shape)

08:15:40  INFO  Scaler fitted on training data only


X_train_scaled shape : (1892508, 38)
X_test_scaled  shape : (473128, 38)


## 2.10 -- SMOTE: Synthetic Minority Oversampling

The training set is 96.49% benign / 3.51% attack.
A model trained directly on this learns to predict benign for everything
and achieves 96.49% accuracy with zero attack detections.

SMOTE generates synthetic attack samples through feature-space interpolation:
for each attack sample, find k=5 nearest neighbours, pick one randomly,
generate a synthetic point along the line segment between them.

Applied on the scaled training set only. Test set is left unchanged
to simulate real-world evaluation at the natural class distribution.

In [31]:
def apply_smote(
    X_train_scaled: pd.DataFrame,
    y_train: pd.Series,
) -> tuple:
    """
    Apply SMOTE oversampling to the scaled training set.

    SMOTE is applied after scaling so that interpolation occurs
    in a normalised feature space, producing more realistic synthetic samples.

    Parameters
    ----------
    X_train_scaled : pd.DataFrame  Scaled training feature matrix.
    y_train        : pd.Series     Binary training labels.

    Returns
    -------
    tuple of (X_train_sm, y_train_sm)
        X_train_sm : pd.DataFrame  SMOTE-balanced training features.
        y_train_sm : pd.Series     Balanced training labels.
    """
    log.info('Applying SMOTE to training set ...')
    log.info('  Before: %s', dict(y_train.value_counts()))

    smote = SMOTE(
        k_neighbors=5,
        random_state=CFG.RANDOM_STATE,
        
    )

    X_sm, y_sm = smote.fit_resample(X_train_scaled, y_train)

    X_train_sm = pd.DataFrame(X_sm, columns=X_train_scaled.columns)
    y_train_sm = pd.Series(y_sm, name='label')

    log.info('  After : %s', dict(pd.Series(y_train_sm).value_counts()))
    log.info('  Training samples after SMOTE: %d', len(X_train_sm))

    return X_train_sm, y_train_sm


X_train_sm, y_train_sm = apply_smote(X_train_scaled, y_train)

print('Training set before SMOTE:')
print('  Benign : {:,}'.format((y_train == 0).sum()))
print('  Attack : {:,}'.format((y_train == 1).sum()))
print()
print('Training set after SMOTE:')
print('  Benign : {:,}'.format((y_train_sm == 0).sum()))
print('  Attack : {:,}'.format((y_train_sm == 1).sum()))
print('  Total  : {:,}'.format(len(y_train_sm)))
print()
print('Test set (unchanged -- real distribution):')
print('  Benign : {:,}'.format((y_test == 0).sum()))
print('  Attack : {:,}'.format((y_test == 1).sum()))
print('  Attack rate: {:.2f}%'.format(y_test.mean() * 100))

08:29:35  INFO  Applying SMOTE to training set ...
08:29:35  INFO    Before: {0: np.int64(1826126), 1: np.int64(66382)}
08:29:46  INFO    After : {0: np.int64(1826126), 1: np.int64(1826126)}
08:29:46  INFO    Training samples after SMOTE: 3652252


Training set before SMOTE:
  Benign : 1,826,126
  Attack : 66,382

Training set after SMOTE:
  Benign : 1,826,126
  Attack : 1,826,126
  Total  : 3,652,252

Test set (unchanged -- real distribution):
  Benign : 456,533
  Attack : 16,595
  Attack rate: 3.51%


## 2.11 -- Visualizations for submission

Five plots saved to `outputs/plots/` at 200 DPI:
1. Class imbalance before and after SMOTE
2. Feature skewness before log transform
3. Before vs after scaling for a representative feature
4. DPI feature correlation with attack label
5. DPI feature distribution: benign vs attack

In [32]:
plt.rcParams.update({
    'figure.dpi'       : 200,
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})


def save_figure(fig: plt.Figure, filename: str) -> None:
    """
    Save a figure to the plots directory at 200 DPI and close it.

    Parameters
    ----------
    fig      : plt.Figure  Figure to save.
    filename : str         Output filename.
    """
    path = CFG.PLOTS / filename
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    log.info('Saved: %s', path)


# Plot 1: Class imbalance before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Class Distribution Before and After SMOTE', fontweight='bold')

before_counts = y_train.value_counts().sort_index()
axes[0].bar(['Benign', 'Attack'], before_counts.values,
            color=['#2c7bb6', '#d7191c'], width=0.5, edgecolor='white')
axes[0].set_title('Training Set -- Before SMOTE')
axes[0].set_ylabel('Sample count')
axes[0].yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: '{:,}'.format(int(x)))
)
for i, count in enumerate(before_counts.values):
    axes[0].text(i, count + before_counts.max() * 0.01,
                '{:,}'.format(count), ha='center', fontsize=9)

after_counts = y_train_sm.value_counts().sort_index()
axes[1].bar(['Benign', 'Attack'], after_counts.values,
            color=['#2c7bb6', '#d7191c'], width=0.5, edgecolor='white')
axes[1].set_title('Training Set -- After SMOTE')
axes[1].set_ylabel('Sample count')
axes[1].yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: '{:,}'.format(int(x)))
)
for i, count in enumerate(after_counts.values):
    axes[1].text(i, count + after_counts.max() * 0.01,
                '{:,}'.format(count), ha='center', fontsize=9)

plt.tight_layout()
save_figure(fig, 'phase2_smote_class_balance.png')


# Plot 2: Feature skewness
raw_X = df[feature_cols].copy().replace([np.inf,-np.inf], np.nan).fillna(0)
original_skew = raw_X.skew().abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = [
    '#d7191c' if v > 2.0 else '#2c7bb6'
    for v in original_skew.values
]
ax.bar(range(len(original_skew)), original_skew.values,
       color=bar_colors, edgecolor='white', linewidth=0.6)
ax.axhline(y=2.0, color='black', linestyle='--', linewidth=1,
           label='Log transform threshold (abs skew = 2.0)')
ax.set_xticks(range(len(original_skew)))
ax.set_xticklabels(original_skew.index, rotation=45, ha='right', fontsize=8)
ax.set_title('Top 20 Feature Skewness Before Log Transform')
ax.set_ylabel('Absolute skewness')
ax.legend(fontsize=9)
save_figure(fig, 'phase2_feature_skewness.png')


# Plot 3: DPI feature correlation with label
dpi_cols_present = [c for c in DPI_FEATURE_COLS if c in df.columns]
corr_cols = dpi_cols_present + ['label']
corr_with_label = (
    df[corr_cols].corr()['label']
    .drop('label')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = [
    '#d7191c' if v > 0 else '#2c7bb6'
    for v in corr_with_label.values
]
ax.barh(corr_with_label.index, corr_with_label.values,
        color=bar_colors, edgecolor='white', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('DPI Feature Correlation with Attack Label')
ax.set_xlabel('Pearson correlation')
ax.invert_yaxis()
save_figure(fig, 'phase2_dpi_correlation.png')


# Plot 4: DPI feature distributions benign vs attack
plot_dpi = [c for c in
            ['burst_score', 'flow_efficiency', 'direction_asymmetry',
             'byte_entropy', 'pkt_size_mean', 'syn_ratio']
            if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    'DPI Feature Distributions -- Benign vs Attack',
    fontsize=12, fontweight='bold',
)
axes = axes.flatten()

for i, col in enumerate(plot_dpi):
    q01 = df[col].quantile(0.01)
    q99 = df[col].quantile(0.99)
    benign = df.loc[df['label'] == 0, col].clip(q01, q99)
    attack = df.loc[df['label'] == 1, col].clip(q01, q99)
    axes[i].hist(benign, bins=50, alpha=0.6, color='#2c7bb6',
                 density=True, label='Benign')
    axes[i].hist(attack, bins=50, alpha=0.6, color='#d7191c',
                 density=True, label='Attack')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'phase2_dpi_distributions.png')

print('All plots saved to outputs/plots/')

08:30:13  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase2_smote_class_balance.png
08:30:23  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase2_feature_skewness.png
08:30:25  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase2_dpi_correlation.png
08:30:30  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase2_dpi_distributions.png


All plots saved to outputs/plots/


## 2.12 -- Preprocessing summary statistics

Structured summary for the paper methods section.

In [27]:
def print_preprocessing_summary(
    df: pd.DataFrame,
    X_train_sm: pd.DataFrame,
    X_test_scaled: pd.DataFrame,
    y_train_sm: pd.Series,
    y_test: pd.Series,
    feature_cols: list,
    flow_features: list,
    dpi_features: list,
    skewed_cols: list,
) -> None:
    """
    Print a structured preprocessing summary for the paper methods section.

    Parameters
    ----------
    df            : pd.DataFrame  Full dataset.
    X_train_sm    : pd.DataFrame  SMOTE-balanced training features.
    X_test_scaled : pd.DataFrame  Scaled test features.
    y_train_sm    : pd.Series     Balanced training labels.
    y_test        : pd.Series     Test labels.
    feature_cols  : list          All feature columns.
    flow_features : list          Raw flow feature columns.
    dpi_features  : list          Engineered DPI feature columns.
    skewed_cols   : list          Log-transformed columns.
    """
    print('=' * 60)
    print('PHASE 2 -- PREPROCESSING SUMMARY')
    print('=' * 60)
    print(f'Total flows             : {len(df):>12,}')
    print(f'Total feature columns   : {len(feature_cols):>12,}')
    print(f'  Flow-level features   : {len(flow_features):>12,}')
    print(f'  Behavioral DPI feats  : {len(dpi_features):>12,}')
    print(f'Log-transformed cols    : {len(skewed_cols):>12,}')
    print(f'Scaling method          : StandardScaler (train-fit only)')
    print(f'Imbalance handling      : SMOTE (training set only)')
    print(f'Split strategy          : Stratified 80/20')
    print(f'Random state            : {CFG.RANDOM_STATE}')
    print()
    print('Training set (after SMOTE):')
    print(f'  Total rows : {len(X_train_sm):>10,}')
    print(f'  Benign     : {(y_train_sm==0).sum():>10,}')
    print(f'  Attack     : {(y_train_sm==1).sum():>10,}')
    print()
    print('Test set (real distribution):')
    print(f'  Total rows : {len(X_test_scaled):>10,}')
    print(f'  Benign     : {(y_test==0).sum():>10,}')
    print(f'  Attack     : {(y_test==1).sum():>10,}')
    print(f'  Attack rate: {y_test.mean()*100:.2f}%')
    print('=' * 60)


print_preprocessing_summary(
    df, X_train_sm, X_test_scaled,
    y_train_sm, y_test,
    feature_cols, flow_features, dpi_features, skewed_cols,
)

PHASE 2 -- PREPROCESSING SUMMARY
Total flows             :    2,365,636
Total feature columns   :           38
  Flow-level features   :           22
  Behavioral DPI feats  :           16
Log-transformed cols    :           17
Scaling method          : StandardScaler (train-fit only)
Imbalance handling      : SMOTE (training set only)
Split strategy          : Stratified 80/20
Random state            : 42

Training set (after SMOTE):
  Total rows :  3,652,252
  Benign     :  1,826,126
  Attack     :  1,826,126

Test set (real distribution):
  Total rows :    473,128
  Benign     :    456,533
  Attack     :     16,595
  Attack rate: 3.51%


## 2.13 -- Save all outputs

Save all processed files needed by Phase 3 and Phase 4.
X_train contains SMOTE-balanced samples. X_test is untouched.

In [34]:
def save_phase2_outputs(
    X_train_sm: pd.DataFrame,
    X_test_scaled: pd.DataFrame,
    y_train_sm: pd.Series,
    y_test: pd.Series,
    y_train_multi: pd.Series,
    y_test_multi: pd.Series,
    scaler: StandardScaler,
    label_encoder: LabelEncoder,
    feature_cols: list,
) -> None:
    """
    Save all Phase 2 outputs to disk.

    X_train.csv contains SMOTE-balanced samples.
    X_test.csv  contains real-distribution samples (unchanged).

    Parameters
    ----------
    X_train_sm    : pd.DataFrame    SMOTE-balanced training features.
    X_test_scaled : pd.DataFrame    Scaled test features.
    y_train_sm    : pd.Series       Balanced binary training labels.
    y_test        : pd.Series       Binary test labels.
    y_train_multi : pd.Series       Multiclass training labels.
    y_test_multi  : pd.Series       Multiclass test labels.
    scaler        : StandardScaler  Fitted scaler.
    label_encoder : LabelEncoder    Fitted encoder.
    feature_cols  : list            Ordered feature column names.
    """
    p = CFG.DATA_PROCESSED

    X_train_sm.to_csv(   p / 'X_train.csv',       index=False)
    X_test_scaled.to_csv(p / 'X_test.csv',        index=False)
    y_train_sm.to_csv(   p / 'y_train.csv',       index=False)
    y_test.to_csv(       p / 'y_test.csv',        index=False)
    y_train_multi.to_csv(p / 'y_train_multi.csv', index=False)
    y_test_multi.to_csv( p / 'y_test_multi.csv',  index=False)

    joblib.dump(scaler,        p / 'scaler.pkl')
    joblib.dump(label_encoder, p / 'label_encoder.pkl')

    pd.Series(feature_cols, name='feature').to_csv(
        p / 'feature_columns.csv', index=False
    )

    log.info('All Phase 2 outputs saved to %s', p)


save_phase2_outputs(
    X_train_sm, X_test_scaled,
    y_train_sm, y_test,
    y_train_multi, y_test_multi,
    scaler, label_encoder, feature_cols,
)

print('=' * 60)
print('PHASE 2 COMPLETE')
print('=' * 60)

files = [
    'X_train.csv', 'X_test.csv',
    'y_train.csv', 'y_test.csv',
    'y_train_multi.csv', 'y_test_multi.csv',
    'scaler.pkl', 'label_encoder.pkl', 'feature_columns.csv',
]
print('Files written to data/processed/:')
for fname in files:
    p = CFG.DATA_PROCESSED / fname
    if p.exists():
        print('  {:<35} {:.2f} MB'.format(fname, p.stat().st_size / 1e6))

g = CFG.DATA_GRAPH / 'graph_data.csv'
if g.exists():
    print()
    print('  {:<35} {:.2f} MB  (data/graph/)'.format(
        'graph_data.csv', g.stat().st_size / 1e6
    ))

print()
print('Feature summary:')
print('  Flow-level features  : {}'.format(len(flow_features)))
print('  DPI features         : {}'.format(len(dpi_features)))
print('  Total features       : {}'.format(len(feature_cols)))
print()
print('Next: run Phase3_Dynamic_Graph.ipynb')

08:37:28  INFO  All Phase 2 outputs saved to C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\data\processed


PHASE 2 COMPLETE
Files written to data/processed/:
  X_train.csv                         1891.34 MB
  X_test.csv                          246.85 MB
  y_train.csv                         10.96 MB
  y_test.csv                          1.42 MB
  y_train_multi.csv                   5.68 MB
  y_test_multi.csv                    1.42 MB
  scaler.pkl                          0.00 MB
  label_encoder.pkl                   0.00 MB
  feature_columns.csv                 0.00 MB

  graph_data.csv                      186.89 MB  (data/graph/)

Feature summary:
  Flow-level features  : 22
  DPI features         : 16
  Total features       : 38

Next: run Phase3_Dynamic_Graph.ipynb
